# Phase 5 - Re-train U-Net on mixed clean + foggy data

**Goal**: build a U-Net that is robust to fog. Strategy: train on a 1:1:1 mix
of (clean VDD, foggy_medium_768 VDD, foggy_dense_768 VDD), then re-evaluate
on all 3 test sets. We expect the new model to:
  - Lose a tiny bit on clean (small sacrifice)
  - Recover most of the mIoU loss on medium fog
  - Recover a significant portion on dense fog

## Phase 4 baseline (numbers to beat)

| Test set            | U-Net v2 (clean only) |
|---------------------|----------------------|
| VDD clean           | 0.7168               |
| VDD foggy medium    | 0.6652  (-7.2%)      |
| VDD foggy dense     | 0.5377  (-25.0%)     |

## Pipeline of this notebook

1. Verify GPU
2. Clone code from GitHub
3. Mount Google Drive
4. Restore datasets from Drive (VDD + 2 foggy variants @768)
5. Install dependencies
6. Launch TensorBoard
7. **Re-train U-Net on mixed data (~60 min)**
8. Backup new run to Drive
9. Evaluate the new model on clean / medium / dense test sets
10. Final comparison table (vs Phase 4 baseline)
11. Backup all results to Drive

**Before running**: Runtime -> Change runtime type -> T4 GPU.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

In [ ]:
import os

if not os.path.isdir('/content/fog-uav-robustness'):
    !git clone https://github.com/FabrizioCozzolino/fog-uav-robustness.git /content/fog-uav-robustness
%cd /content/fog-uav-robustness
!ls

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fog-uav-robustness'

## 4. Restore datasets from Drive

We need 3 datasets: clean VDD + foggy_medium_768 + foggy_dense_768. The 256
variants are not used (Phase 4 showed they're contaminated by upscaling artifacts).

If your Phase 4 session is still alive, this cell will skip everything since
the data is already in /content/data/. Otherwise it copies fresh from Drive.

In [ ]:
import os, time, shutil

DATASETS_NEEDED = ['VDD', 'VDD_foggy_medium_768', 'VDD_foggy_dense_768']

for ds in DATASETS_NEEDED:
    LOCAL = f'/content/data/{ds}'
    train_src = os.path.join(LOCAL, 'train', 'src')
    if os.path.isdir(train_src) and len(os.listdir(train_src)) >= 280:
        n = len(os.listdir(train_src))
        print(f'[ok] {ds} already at {LOCAL} ({n} train images)')
        continue

    DRIVE_DS = f'{DRIVE}/data/{ds}'
    # Detect possible nested layout for VDD only
    if ds == 'VDD' and os.path.isdir(os.path.join(DRIVE_DS, 'VDD', 'train')):
        SRC = os.path.join(DRIVE_DS, 'VDD')
    elif os.path.isdir(os.path.join(DRIVE_DS, 'train')):
        SRC = DRIVE_DS
    else:
        raise RuntimeError(f'Cannot find train/ inside {DRIVE_DS}')

    if os.path.isdir(LOCAL):
        shutil.rmtree(LOCAL)
    os.makedirs('/content/data', exist_ok=True)
    print(f'Copying {SRC} -> {LOCAL}')
    t0 = time.time()
    !cp -r "{SRC}" "{LOCAL}"
    print(f'  done in {time.time() - t0:.0f} s')

print('\n=== File counts ===')
for ds in DATASETS_NEEDED:
    LOCAL = f'/content/data/{ds}'
    counts = {split: len(os.listdir(f'{LOCAL}/{split}/src'))
              for split in ['train','val','test']}
    print(f'  {ds}: train={counts["train"]}, val={counts["val"]}, test={counts["test"]}')

## 5. Install Python dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 6. Launch TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/runs

## 7. Re-train U-Net on mixed data (clean + medium + dense)

Configuration:
- Mixed training set: clean (280) + medium (280) + dense (280) = **840 images**
- Validation: clean only (we use clean val to track convergence on the 'reference')
- Same hyperparameters as Phase 1 v2: 768x768, batch 4, AdamW 1e-4, cosine annealing, class weights inverse_sqrt
- 30 epochs (matches Phase 1 for fair comparison; could be increased)

Estimated time on T4: ~60 minutes (3x more data per epoch than Phase 1).

*Why this hyperparameter setup?* We want to isolate the effect of the data
augmentation: same model, same optimizer, same schedule, only the training
data differs. This is the cleanest experimental design.

In [ ]:
!python src/training/train_unet.py \
    --data-root /content/data/VDD \
    --data-roots /content/data/VDD \
                 /content/data/VDD_foggy_medium_768 \
                 /content/data/VDD_foggy_dense_768 \
    --output-dir outputs/runs \
    --run-name unet_resnet34_mixed_v3 \
    --encoder resnet34 \
    --image-size 768 \
    --epochs 30 \
    --batch-size 4 \
    --val-batch-size 4 \
    --num-workers 2 \
    --lr 1e-4 \
    --weight-decay 1e-4 \
    --grad-clip 1.0 \
    --class-weights inverse_sqrt \
    --seed 42

## 8. Backup new run to Drive

In [ ]:
import shutil, os

RUN_NAME = 'unet_resnet34_mixed_v3'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN_NAME}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN_NAME}'
if os.path.isdir(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print(f'[ok] backed up to Drive: {DST}')
!ls -la "{DST}"

## 9. Evaluate v3 on the 3 test sets

Each evaluation takes ~15-30 seconds on T4.

In [ ]:
# Eval on VDD clean
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_mixed_v3/best.pth \
    --data-root /content/data/VDD \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_mixed_v3/test_results_clean.json

In [ ]:
# Eval on VDD foggy medium 768
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_mixed_v3/best.pth \
    --data-root /content/data/VDD_foggy_medium_768 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_mixed_v3/test_results_foggy_medium_768.json \
    --no-tb

In [ ]:
# Eval on VDD foggy dense 768
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_mixed_v3/best.pth \
    --data-root /content/data/VDD_foggy_dense_768 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_mixed_v3/test_results_foggy_dense_768.json \
    --no-tb

## 10. Final comparison table

Compare U-Net v2 (clean training) vs U-Net v3 (mixed training) on all 3 test
sets. This is **the** result of the project.

In [ ]:
import json, os

V2 = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_clean_v2_weighted'
V3 = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_mixed_v3'

test_sets = [
    ('clean',       'test_results_clean.json'),
    ('medium @768', 'test_results_foggy_medium_768.json'),
    ('dense @768',  'test_results_foggy_dense_768.json'),
]

def load_or_none(path):
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        return json.load(f)

v2_results = {label: load_or_none(f'{V2}/{fname}') for label, fname in test_sets}
v3_results = {label: load_or_none(f'{V3}/{fname}') for label, fname in test_sets}

print('=' * 90)
print('PHASE 5: U-Net robustness, before vs after foggy data augmentation')
print('=' * 90)
print(f'{"test set":12s}  {"v2 mIoU":>9s}  {"v3 mIoU":>9s}  {"v3 - v2":>10s}  {"recovered?":>14s}')
print('-' * 90)
for label, _ in test_sets:
    v2 = v2_results.get(label)
    v3 = v3_results.get(label)
    if v2 is None or v3 is None:
        print(f'{label:12s}  [missing data]')
        continue
    delta = v3['mIoU'] - v2['mIoU']
    if 'foggy' in v2.get('checkpoint', '') or 'foggy' in label:
        clean_baseline = v2_results.get('clean', {}).get('mIoU', None)
        if clean_baseline:
            recovery = (v3['mIoU'] - v2['mIoU']) / max(clean_baseline - v2['mIoU'], 1e-9) * 100
            recovery_str = f'{recovery:+.1f}%'
        else:
            recovery_str = '-'
    else:
        recovery_str = '-'
    print(f'{label:12s}  {v2["mIoU"]:>9.4f}  {v3["mIoU"]:>9.4f}  {delta:>+10.4f}  {recovery_str:>14s}')

print()
print('Per-class IoU comparison:')
print('-' * 90)
header = f'{"class":14s}'
for label, _ in test_sets:
    header += f'  {label+" v2":>14s}  {label+" v3":>14s}'
print(header)
if v2_results.get('clean') and v3_results.get('clean'):
    classes = list(v2_results['clean']['per_class_iou'].keys())
    for cls in classes:
        row = f'{cls:14s}'
        for label, _ in test_sets:
            v2 = v2_results.get(label)
            v3 = v3_results.get(label)
            if v2 and v3:
                row += f'  {v2["per_class_iou"][cls]:>14.4f}  {v3["per_class_iou"][cls]:>14.4f}'
            else:
                row += f'  {"--":>14s}  {"--":>14s}'
        print(row)

In [ ]:
# Visualize the v2 vs v3 comparison
import matplotlib.pyplot as plt
import numpy as np

if all(v2_results.get(l) and v3_results.get(l) for l, _ in test_sets):
    labels = [l for l, _ in test_sets]
    v2_miou = [v2_results[l]['mIoU'] for l in labels]
    v3_miou = [v3_results[l]['mIoU'] for l in labels]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width/2, v2_miou, width, label='U-Net v2 (clean training)', color='#3498DB')
    bars2 = ax.bar(x + width/2, v3_miou, width, label='U-Net v3 (mixed training)', color='#27AE60')

    ax.set_ylabel('test mIoU')
    ax.set_title('U-Net robustness recovery via foggy data augmentation')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1)

    # Annotate values
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    out = '/content/fog-uav-robustness/outputs/figures/phase5_v2_vs_v3.png'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    plt.savefig(out, dpi=80, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {out}')

## 11. Backup final results to Drive

In [ ]:
import shutil, os

# Copy any test_results_*.json from v3 run to Drive
RUN = 'unet_resnet34_mixed_v3'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN}'
for fname in os.listdir(SRC):
    if fname.startswith('test_results') and fname.endswith('.json'):
        src_f = os.path.join(SRC, fname)
        dst_f = os.path.join(DST, fname)
        shutil.copy(src_f, dst_f)
        print(f'[ok] {fname}')

# Re-back up TB folder so the new training curves are saved
TB_SRC = f'{SRC}/tb'
TB_DST = f'{DST}/tb'
if os.path.isdir(TB_SRC):
    if os.path.isdir(TB_DST):
        shutil.rmtree(TB_DST)
    shutil.copytree(TB_SRC, TB_DST)
    print('[ok] tb/ backed up')

# Backup figure
FIG_SRC = '/content/fog-uav-robustness/outputs/figures/phase5_v2_vs_v3.png'
FIG_DST = '/content/drive/MyDrive/fog-uav-robustness/outputs/figures/phase5_v2_vs_v3.png'
if os.path.isfile(FIG_SRC):
    os.makedirs(os.path.dirname(FIG_DST), exist_ok=True)
    shutil.copy(FIG_SRC, FIG_DST)
    print('[ok] figure backed up')

print('\nAll done!')
!ls -la "{DST}" | grep -E 'test_results|history|best'

## What's next (Phase 6)

All experimental results are now on Drive. The remaining work is:

1. Write the project report (LaTeX or Word) with:
   - Introduction and motivation (UAV semantic segmentation, fog robustness problem)
   - Methodology (Pix2Pix domain transfer + U-Net training)
   - Results: Phase 1 baseline -> Phase 2 GAN training -> Phase 3 generation -> Phase 4 robustness drop -> Phase 5 recovery via augmentation
   - Discussion of limits (Cityscapes domain != UAV domain, depth-based fog model on aerial imagery)
   - Conclusions

2. Prepare the oral defense slides.

3. Practice answering exam questions about each design choice.